<a href="https://colab.research.google.com/github/eisbetterthanpi/vision/blob/main/hydra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title data
import torch
import torchvision
import torchvision.transforms as transforms
# transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
transform = transforms.Compose([transforms.ToTensor(),])

# CIFAR10: 60000 32x32 color images in 10 classes, with 6000 images per class
train_dataset = torchvision.datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='data', train=False, download=True, transform=transform)
batch_size = 128 # 4
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

import numpy as np
import matplotlib.pyplot as plt
def imshow(img):
    # img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# dataiter = iter(train_loader) # get some random training images
# images, labels = next(dataiter)
# print(images.shape) # [batch, 3, 32, 32]
# imshow(torchvision.utils.make_grid(images))


100%|██████████| 170M/170M [00:30<00:00, 5.66MB/s]


In [2]:
# @title ssd me
# https://goombalab.github.io/blog/2024/mamba2-part4-systems/
# https://tridao.me/blog/2024/mamba2-part3-algorithm/
# # https://github.com/state-spaces/mamba/blob/main/mamba_ssm/ops/triton/ssd_combined.py
import torch
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def segsum(x): # [...,c] # Naive segment sum calculation. exp(segsum(A)) produces a 1-SS matrix, which is equivalent to a scalar SSM.
    T = x.size(-1)
    x_cumsum = torch.cumsum(x, dim=-1)
    x_segsum = x_cumsum.unsqueeze(-1) - x_cumsum.unsqueeze(-2) # [...,c,c] # vert-hori
    mask = torch.triu(torch.ones(T, T, device=x.device, dtype=bool), diagonal=1)
    return x_segsum.masked_fill(mask, -torch.inf) # [...,c,c]

# def ssd(X, A, B, C, h0=None, b_ind=None, chunk=64): # X:[b,h,t,d], A:[b,h,t], B/C:[b,h,t,s], h0:[b,h,d,s], b_ind:[b]
def ssd(X, A, B, C, h0=None, msk=None, b_ind=None, chunk=64): # X:[b,h,t,d], A:[b,h,t], B/C:[b,h,t,s], h0:[b,h,d,s], msk:[b,t], b_ind:[b]
    assert X.dtype == A.dtype == B.dtype == C.dtype
    assert X.shape[:-1] == A.shape == B.shape[:-1]
    assert h0==None or b_ind==None
    # print('ssd', X.shape, A.shape, B.shape)
    if b_ind!=None: A[:,:,b_ind[1:-1]] = 0 # at boundaries, A=0
    if X.shape[2] % chunk != 0: X, A, B, C = [x.unsqueeze(2) for x in (X, A, B, C)]
    else: X, A, B, C = [x.unflatten(2, (-1,chunk)) for x in (X, A, B, C)] # [b,h,t/c,c(,d/s)]

    # 1. Compute the output for each intra-chunk (diagonal blocks)
    L = torch.exp(segsum(A)) # [b,h,t/c,c,c]
    # if msk!=None: L[msk] = 0 # L = L.masked_fill(msk, 0)
    if msk!=None: # this only saves computation from ssd. full info leakage, full compute in in_proj
        b,h,l,c,c = L.shape
        assert msk.shape==(b,l*c)
        msk = msk.reshape(b,1,l,c)
        msk = msk.unsqueeeze(-2) | msk.unsqueeze(-1) # [b,1,t/c,c,c]
        L[msk] = 0 # L = L.masked_fill(msk, 0)
    Y_diag  = torch.einsum("...cs,...ks,...ck,...kd->...cd", C, B, L, X) # bhlcs,bhlks,bhlck,bhlkd->bhlcd # full CA...ABX? for chunks

    # 2. Compute the state for each intra-chunk # (right term of low-rank factorization of off-diagonal blocks; B terms)
    A_cumsum = torch.cumsum(A, dim=-1) # [b,h,t/c,c]
    decay_states = torch.exp((A_cumsum[...,-1:] - A_cumsum)) # [b,h,t/c,c] # Ai+1...T
    states = torch.einsum("...cs,...c,...cd->...ds", B, decay_states, X) # bhlcs,bhlc,bhlcd->bhlds # BiXiAi+1...T

    # 3. Compute the inter-chunk SSM recurrence; produces correct SSM states at chunk boundaries # (middle term of factorization of off-diag blocks; A terms)
    if h0==None: h0 = torch.zeros_like(states[:,:,0], device=states.device) # [b,h,d,s]
    states = torch.cat([h0.unsqueeze(2), states], dim=2) # [b,1+t/c,h,d,s] # h0,hc,h2c,...,ht
    decay_chunk = torch.exp(segsum(F.pad(A_cumsum[...,-1], (1,0)))) # [b,h,1+t/c]-> # [b,h,1+t/c,1+t/c] # 1,A1...1t/c,A1t/c+1...2t/c,...,A(c-1)t/c...At
    new_states = torch.einsum("...tl,...lds->...tds", decay_chunk, states) # bhtl,bhlds->bhtds # h0, BiXi/A1...i-1,

    # 4. Compute state -> output conversion per chunk # (left term of low-rank factorization of off-diagonal blocks; C terms)
    Y_off = torch.einsum('...cs,...ds,...c->...cd', C, new_states[:,:,:-1], torch.exp(A_cumsum)) # bhlcs,bhlds,bhlc->bhlcd # offset for each chunk # C1h0A1, Ci BiXi/A1...i-1,
    Y = (Y_diag+Y_off).flatten(2,3)
    return Y, new_states[:,:,-1] # [b,t,h,d], [b,h,d,s]


In [ ]:
# @title Hydra me
# https://github.com/goombalab/hydra/blob/main/hydra/modules/hydra.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def zero_module(module):
    for p in module.parameters():
        p.detach().zero_()
    return module

class Hydra(nn.Module):
    def __init__(self, d_model, expand=2, n_heads=8, n_groups=8, d_state=8, d_conv=7):
        super().__init__()
        n_groups = min(n_heads, n_groups)
        self.d_model, self.n_groups, self.d_state, self.d_conv = d_model, n_groups, d_state, d_conv
        self.d_inner = expand * self.d_model
        self.n_heads, self.d_head = n_heads, self.d_inner//n_heads

        self.in_proj = nn.Linear(self.d_model, 2* self.d_inner + 2* (2*self.n_groups*self.d_state) + 2* self.n_heads, bias=False) # z,x,B,C,dt
        conv_dim = self.d_inner + 2* (2*self.n_groups*self.d_state) # for x,B,C
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=d_conv, groups=conv_dim, padding=d_conv//2, bias=True)
        # self.conv1d.weight._no_weight_decay = True
        self.act = nn.SiLU()

        # self.h0 = nn.Parameter(torch.zeros(self.n_heads, self.d_head, self.d_state))
        # self.h0._no_weight_decay = True
        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.n_heads) * (math.log(dt_max)-math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_weight_decay = True
        A = torch.empty(self.n_heads, dtype=torch.float32).uniform_(1,16)
        # A = torch.empty(self.n_heads).uniform_(1,16)
        # A = torch.ones(self.n_heads, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A))
        self.A_log._no_weight_decay = True

        self.D = nn.Linear(self.d_inner, self.n_heads)
        # nn.init.constant_(self.D.weight, 1)
        self.D.bias._no_weight_decay = True

        self.norm = nn.RMSNorm(self.d_inner)
        self.out = zero_module(nn.Linear(self.d_inner, self.d_model, bias=False))

    # def forward(self, u, h=None): # [b,t,d]
    def forward(self, u, h=None, b_ind=None): # [b,t,d]
        b = u.shape[0]
        # print('Hydra u', u.shape)
        A = -torch.exp(self.A_log) # [n_heads]
        z, xBC, dt = self.in_proj(u).split([self.d_inner, self.d_inner + 2* (2*self.n_groups*self.d_state), 2* self.n_heads], dim=-1)
        dt = torch.cat((dt[...,:self.n_heads], torch.flip(dt[...,self.n_heads:], (1,))), dim=0) # [b,t,2h]->[2b,t,h]
        dt = F.softplus(dt+self.dt_bias) # [2b,t,h]+[h]
        xBC = self.act(self.conv1d(xBC.transpose(-2,-1)).transpose(-2,-1))  # [b,t, d_inner + 2* n_groups*d_state]
        x, BC = xBC.split([self.d_inner, 2* (2*self.n_groups*self.d_state)], dim=-1)

        x_og = x # [b,t,inr]
        x = torch.cat((x, torch.flip(x,(1,))), dim=0).unflatten(-1, (self.n_heads, self.d_head)) # [2b,t,inr]->[2b,t,h,d]
        B, C = torch.cat((BC[...,:2*self.n_groups*self.d_state], torch.flip(BC[...,2*self.n_groups*self.d_state:], (1,))), dim=0).chunk(2, dim=-1) # [b,t,2*2gs]->[2b,t,2gs]->[2b,t,gs]
        B, C = B.unflatten(-1, (self.n_groups, self.d_state)), C.unflatten(-1, (self.n_groups, self.d_state)) # x:[2b,t,h,d], B/C:[2b,t,g,s]
        h_g = self.n_heads//self.n_groups
        if h_g>1: B, C = B.repeat_interleave(h_g, dim=-2), C.repeat_interleave(h_g, dim=-2) # [b,t,g,s]->[b,t,h,s]
        x, B, C, dt = [a.transpose(1,2) for a in (x, B, C, dt)] # X:[b,h,t,d], B/C:[b,h,t,s], dt:[b,h,t]

        # h0 = self.h0.expand(u.size(0),-1,-1,-1) # [b,n,d,s]
        # print('x dt a b', x.shape, dt.shape, A.shape, B.shape)
        y, h = ssd(x*dt.unsqueeze(-1), A.unsqueeze(-1)*dt, B, C, h, b_ind, 64) # 256
        y = y.transpose(1,2).flatten(2) # [2b,t,d]/[1,2b*t,d]

        y = torch.roll(y, shifts=1, dims=1) # 123...l -> l12...l-1
        y[:,0] = 0 # 012...l-1
        y = y[:b] + torch.flip(y[b:], (1,)) + x_og * self.D(x_og).repeat(1,1,self.d_head) # [b,t,h*inr]

        y = self.norm(y * self.act(z)) # [b,t,d_inner] # norm(x)*silu(z) if norm_before_gate, else norm(x*silu(z)) # https://github.com/state-spaces/mamba/blob/main/mamba_ssm/ops/triton/layernorm_gated.py#L18
        # y = self.norm(y) * self.act(z)
        out = self.out(y)
        return out, h # [b,t,in]


b,t,d_model=5,256,32
# b,t,d_model=5,7,32
u = torch.randn(b,t,d_model, device=device)
model = Hydra(d_model).to(device)
print(sum(p.numel() for p in model.parameters() if p.requires_grad)) #
out, h = model(u)
# h0 = torch.randn(b, model.n_heads, model.d_head, model.d_state)
# print(out.shape)
# print(out.shape, h.shape)
# print(out[0,-3:,:5], h[0,:2,:5,:5])
# out, h = model(u, h)

print(out.shape)
# print(out[0,-3:,:5], h[0,:2,:5,:5])


In [ ]:
# @title Hydra one BCdt
# https://github.com/goombalab/hydra/blob/main/hydra/modules/hydra.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def zero_module(module):
    for p in module.parameters():
        p.detach().zero_()
    return module

# @torch.compile()
class Hydra(nn.Module):
    # def __init__(self, d_model, expand=2, n_heads=8, n_groups=8, d_state=8, d_conv=7):
    def __init__(self, d_model, expand=3, n_heads=8, n_groups=8, d_state=8, d_conv=7):
    # def __init__(self, d_model, expand=4, n_heads=8, n_groups=8, d_state=8, d_conv=7):
        super().__init__()
        n_groups = min(n_heads, n_groups)
        self.d_model, self.n_groups, self.d_state, self.d_conv = d_model, n_groups, d_state, d_conv
        self.d_inner = expand * self.d_model
        self.n_heads, self.d_head = n_heads, self.d_inner//n_heads

        self.in_proj = nn.Linear(self.d_model, 2* self.d_inner + 2*self.n_groups*self.d_state + self.n_heads, bias=False) # z,x,B,C,dt
        conv_dim = self.d_inner + 2*self.n_groups*self.d_state # for x,B,C
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=d_conv, groups=conv_dim, padding=d_conv//2, bias=True)
        # self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=d_conv, groups=conv_dim, padding=d_conv-1, bias=True)
        # self.conv1d.weight._no_weight_decay = True
        self.act = nn.SiLU()

        # self.h0 = nn.Parameter(torch.zeros(self.n_heads, self.d_head, self.d_state))
        # self.h0._no_weight_decay = True
        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.n_heads) * (math.log(dt_max)-math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_weight_decay = True
        A = torch.empty(self.n_heads, dtype=torch.float32).uniform_(1,16)
        # A = torch.empty(self.n_heads).uniform_(1,16)
        # A = torch.ones(self.n_heads, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A))
        self.A_log._no_weight_decay = True

        self.D = nn.Linear(self.d_inner, self.n_heads)
        # nn.init.constant_(self.D.weight, 1)
        self.D.bias._no_weight_decay = True

        self.norm = nn.RMSNorm(self.d_inner)
        self.out = zero_module(nn.Linear(self.d_inner, self.d_model, bias=False))

    # def forward(self, u, h=None): # [b,t,d]
    def forward(self, u, h=None, b_ind=None): # [b,t,d]
        b = u.shape[0]
        # print('Hydra u', u.shape)
        A = -torch.exp(self.A_log) # [n_heads]
        z, xBC, dt = self.in_proj(u).split([self.d_inner, self.d_inner + 2*self.n_groups*self.d_state, self.n_heads], dim=-1)
        dt = torch.cat((dt, torch.flip(dt,(1,))), dim=0) # [b,t,h]->[2b,t,h]
        dt = F.softplus(dt+self.dt_bias) # [2b,t,h]+[h]
        xBC = self.act(self.conv1d(xBC.transpose(-2,-1)).transpose(-2,-1))  # [b,t, d_inner + 2* n_groups*d_state]
        x, BC = xBC.split([self.d_inner, 2*self.n_groups*self.d_state], dim=-1)

        x_og = x # [b,t,inr]
        x = torch.cat((x, torch.flip(x,(1,))), dim=0).unflatten(-1, (self.n_heads, self.d_head)) # [2b,t,inr]->[2b,t,h,d]
        B, C = torch.cat((BC, torch.flip(BC,(1,))), dim=0).chunk(2, dim=-1) # [b,t,2gs]->[2b,t,2gs]->[2b,t,gs]
        B, C = B.unflatten(-1, (self.n_groups, self.d_state)), C.unflatten(-1, (self.n_groups, self.d_state)) # x:[2b,t,h,d], B/C:[2b,t,g,s]
        h_g = self.n_heads//self.n_groups
        if h_g>1: B, C = B.repeat_interleave(h_g, dim=-2), C.repeat_interleave(h_g, dim=-2) # [b,t,g,s]->[b,t,h,s]
        x, B, C, dt = [a.transpose(1,2) for a in (x, B, C, dt)] # X:[b,h,t,d], B/C:[b,h,t,s], dt:[b,h,t]

        # h0 = self.h0.expand(u.size(0),-1,-1,-1) # [b,n,d,s]
        # print('x dt a b', x.shape, dt.shape, A.shape, B.shape)
        y, h = ssd(x*dt.unsqueeze(-1), A.unsqueeze(-1)*dt, B, C, h, b_ind, 64) # 256
        y = y.transpose(1,2).flatten(2) # [2b,t,d]/[1,2b*t,d]

        y = torch.roll(y, shifts=1, dims=1) # 123...l -> l12...l-1
        y[:,0] = 0 # 012...l-1
        y = y[:b] + torch.flip(y[b:], (1,)) + x_og * self.D(x_og).repeat(1,1,self.d_head) # [b,t,h*inr]

        y = self.norm(y * self.act(z)) # [b,t,d_inner] # norm(x)*silu(z) if norm_before_gate, else norm(x*silu(z)) # https://github.com/state-spaces/mamba/blob/main/mamba_ssm/ops/triton/layernorm_gated.py#L18
        # y = self.norm(y) * self.act(z)
        out = self.out(y)
        return out, h # [b,t,in]


b,t,d_model=5,256,32
# b,t,d_model=5,7,32
u = torch.randn(b,t,d_model, device=device)
model = Hydra(d_model).to(device)
print(sum(p.numel() for p in model.parameters() if p.requires_grad)) # 59850
out, h = model(u)
# h0 = torch.randn(b, model.n_heads, model.d_head, model.d_state)
# print(out.shape)
# print(out.shape, h.shape)
# print(out[0,-3:,:5], h[0,:2,:5,:5])
# out, h = model(u, h)

print(out.shape)
# print(out[0,-3:,:5], h[0,:2,:5,:5])

# noroll is still valid
# increase d_state to d_model*expand//n_groups? = d_inner//n_groups
# causal conv by slicing


In [3]:
# @title euler zoh bilinear diagonal A
import torch

# Euler: Abar = 1 + ∆A ; Bbar = ∆B
def euler(A, B, dt): # bts, btsd, bt / bths, bthsd, bt
    # return 1+dt.unsqueeze(-1)*A, dt[...,None,None]*B
    return 1+torch.einsum('bt,bt...s->bt...s', dt, A), torch.einsum('bt,bt...sd->bt...sd', dt, B)

# ZOH: Abar = exp(∆A) ; Bbar = (exp(∆A)-I)/A · B ~ (1+A/2)B # For numerical stability as A->0 # (e^x -1)/x ~ 1+(x/2) # from taylor expansion, e^x ~ 1 + x + x^2/2 + x^3/6 + x^4/24 + ...
def zoh(A, B, dt): # btd, btds, bt
    # A_bar = torch.exp(dt.unsqueeze(-1)*A)
    A_bar = torch.exp(torch.einsum('bt,bt...s->bt...s', dt, A))
    return A_bar, ((A_bar-1)/A).unsqueeze(-1) * B
    # return torch.exp(dt.unsqueeze(-1)*A), (1+A/2).unsqueeze(-1) * B # For numerical stability

# Bilinear: Abar = (1+∆A/2) / (1-∆A/2) ; Bbar = ∆B/(1-∆A/2)
def bilinear(A, B, dt): # btd, btds, bt
    # dA_2 = dt.unsqueeze(-1)*A/2
    dA_2 = torch.einsum('bt,bt...s->bt...s', dt, A/2)
    # return (1+dA_2)/(1-dA_2), (dt.unsqueeze(-1)/(1-dA_2)).unsqueeze(-1) * B
    return (1+dA_2)/(1-dA_2), torch.einsum('bt,bt...s->bt...s', dt, 1-dA_2).unsqueeze(-1) * B

b,t,d,s = 2,5,8,4
h=3
A = -torch.randn(b,t,d) # Negative for stability?
B = torch.randn(b,t,d,s)
dt = torch.rand(b,t)*.1
# A_bar, B_bar = euler(A, B, dt)
# A_bar, B_bar = zoh(A, B, dt)
A_bar, B_bar = bilinear(A, B, dt)
print(A_bar.shape, B_bar.shape)

A = -torch.randn(b,t,h,d) # Negative for stability?
B = torch.randn(b,t,h,d,s)

# A_bar, B_bar = euler(A, B, dt)
# print(A_bar.shape, B_bar.shape)
# A_bar, B_bar = zoh(A, B, dt)
# print(A_bar.shape, B_bar.shape) # [2, 5, 8, 8], [2, 5, 8, 4]
A_bar, B_bar = bilinear(A, B, dt)
print(A_bar.shape, B_bar.shape) # [2, 5, 8, 8], [2, 5, 8, 4]


torch.Size([2, 5, 8]) torch.Size([2, 5, 8, 4])
torch.Size([2, 5, 3, 8]) torch.Size([2, 5, 3, 8, 4])


In [4]:
# @title Mamba2 me
# https://github.com/state-spaces/mamba/blob/main/mamba_ssm/modules/mamba2_simple.py
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def zero_module(module):
    for p in module.parameters():
        p.detach().zero_()
    return module

# @torch.compile()
class Hydra(nn.Module):
    def __init__(self, d_model, expand=3, n_heads=8, n_groups=8, d_state=8, d_conv=7):
        super().__init__()
        n_groups = min(n_heads, n_groups)
        assert n_heads % n_groups == 0, "nheads must be divisible by ngroups"
        self.d_model, self.n_groups, self.d_state, self.d_conv = d_model, n_groups, d_state, d_conv
        self.d_inner = expand * self.d_model
        self.n_heads, self.d_head = n_heads, self.d_inner//n_heads

        self.in_proj = nn.Linear(self.d_model, 2* self.d_inner + 2* self.n_groups*self.d_state + self.n_heads, bias=False) # z,x,B,C,A
        # with torch.no_grad(): self.in_proj.weight[self.d_inner:] = .1*self.in_proj.weight[self.d_inner:]
        conv_dim = self.d_inner + 2*self.n_groups*self.d_state # for x,B,C
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=d_conv, groups=conv_dim, padding=d_conv//2, bias=True)
        self.act = nn.SiLU()

        # self.h0 = nn.Parameter(torch.zeros(self.n_heads, self.d_head, self.d_state))
        # self.D = nn.Parameter(torch.ones(self.n_heads)) # from mamba
        self.D = nn.Linear(self.d_inner, self.n_heads) # og
        self.norm = nn.RMSNorm(self.d_inner)
        self.out = zero_module(nn.Linear(self.d_inner, self.d_model, bias=False))

    def forward(self, u, h=None, mask=None, b_ind=None): # [b,t,d], ([b,k-1,xbc], [b,h,d,s]), [b+1]
        b, t = u.shape[:2]
        step = torch.ones(2*b,t, device=u.device) # [b,t]
        # step = torch.ones(*u.shape[:2], device=u.device) # [b,t]
        z, xBC, A = self.in_proj(u).split([self.d_inner, self.d_inner + 2*self.n_groups*self.d_state, self.n_heads], dim=-1)
        A = torch.cat((A, torch.flip(A,(1,))), dim=0) # [b,t,h]->[2b,t,h]
        # A = -torch.exp(A) #
        # A = -F.softplus(-A) # log(1+e^x)
        A = -F.sigmoid(-A) # 1/(1+e^-x)
        # print('mamba fwd0', A[-1,-3:,-5:])

        if h==None: h_conv, h_ssm = None, None
        else: h_conv, h_ssm = h
        # print('mamba fwd', xBC.shape, h_conv.shape)
        xBC = self.act(self.conv1d(xBC.transpose(-2,-1)).transpose(-2,-1)) # [b,t,inr+2gs]

        xBC = torch.cat((xBC, torch.flip(xBC,(1,))), dim=0) # [b,t,xbc]->[2b,t,xbc]
        x, B, C = xBC.split([self.d_inner, self.n_groups*self.d_state, self.n_groups*self.d_state], dim=-1)

        x_og = x[:b] # [b,t,inr]
        x, B, C = x.unflatten(-1, (self.n_heads, self.d_head)), B.unflatten(-1, (self.n_groups, self.d_state)), C.unflatten(-1, (self.n_groups, self.d_state)) # x:[2b,t,h,d], B/C:[2b,t,g,s]
        # y_diag = (x[:b] * self.D.unsqueeze(-1)).flatten(2) # bthd*h1

        h_g = self.n_heads//self.n_groups
        if h_g>1: B, C = B.repeat_interleave(h_g, dim=-2), C.repeat_interleave(h_g, dim=-2) # [b,t,g,s]->[b,t,h,s]

        # A, x = (A*step[...,*[None]*(A.ndim-2)]).exp(), x*step[...,None,None] # bthd*bt11=bthd # sdd discretization
        A, B = zoh(A, B, step) # euler zoh bilinear # bth,bths
        # print('mamba fwd1', A[-1,-3:,-5:])

# x:bthd, dt:bthd, A.exp:hds, B/C:bts, D:hd # mamba1
# x:bhtd, dt:bht, A:h, B/C:bgts-repeat>bhts, D:h # mamba2
# mamba(ssd):
# h = Ah + Bx : A*h + x@B = 1/ds*ds + d1@1s = ds
# y = Ch + Dx : h@C + D*x = ds@s1 + 1/d1*d1 = d1

        # print('mamba fwd', x.shape, A.shape, B.shape)
        x, A, B, C = [a.transpose(1,2) for a in (x, A, B, C)] # X:[b,h,t,d], A:[b,h,t], B/C:[b,h,t,s]
# x:bhtd, A:bht, B/C:bhts, 10.5s
        y, h_ssm = ssd(x, A.log(), B, C, h_ssm, msk=mask, b_ind=b_ind, chunk=64) #
        y = y.transpose(1,2).flatten(2) # [b,t,d]/[1,b*t,d]

        y = torch.roll(y, shifts=1, dims=1) # 123...l -> l12...l-1
        y[:,0] = 0 # 012...l-1
        y = y[:b] + torch.flip(y[b:], (1,)) + x_og * self.D(x_og).repeat(1,1,self.d_head) # [b,t,inr]
        # y = y[:b] + torch.flip(y[b:], (1,)) +  y_diag # [b,t,inr]

        y = self.norm(y * self.act(z)) # [b,t,inr] # norm(x)*silu(z) if norm_before_gate, else norm(x*silu(z)) # https://github.com/state-spaces/mamba/blob/main/mamba_ssm/ops/triton/layernorm_gated.py#L18
        out = self.out(y)
        return out, (h_conv, h_ssm) # [b,t,in], ([b,k-1,xbc], [b,h,d,s])


b,t,d_model=5,256,32
# b,t,d_model=5,7,32
u = torch.randn(b,t,d_model, device=device)
model = Hydra(d_model).to(device)
print(sum(p.numel() for p in model.parameters() if p.requires_grad)) #
out, h = model(u)
# h0 = torch.randn(b, model.n_heads, model.d_head, model.d_state)
# print(out.shape)
# print(out.shape, h.shape)
# print(out[0,-3:,:5], h[0,:2,:5,:5])
# out, h = model(u, h)

print(out.shape)
# print(out[0,-3:,:5], h[0,:2,:5,:5])


16232
torch.Size([5, 256, 32])


In [ ]:
print(model.blk[0].rnn.A_log)
# [1.2613, 0.0455, 1.1601, 0.8309, 1.5953, 1.7973, 1.1598, 2.2607]

In [5]:
# @title Hydra Block ViT
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import inspect
class Seq(nn.Sequential):
    def __init__(self, *args):
        super().__init__(*args)
    def forward(self, x, hid=None):
        hidden = []
        for i, layer in enumerate(self):
            x, h = layer(x, hid[i] if hid!=None else None)
            hidden.append(h)
        return x, hidden

class Block(nn.Module):
    def __init__(self, d_model, n_heads, drop=0):
        super().__init__()
        self.d_model = d_model
        self.norm1 = nn.RMSNorm(d_model) # LayerNorm RMSNorm
        # self.norm2 = nn.RMSNorm(d_model)
        # self.drop = nn.Dropout(drop)
        self.rnn = Hydra(d_model, n_heads=n_heads)

    def forward(self, x, h=None): # [b,t,d], (,)
        # print('blk fwd', x.shape, h.shape if h is not None else None)
        x_, h = self.rnn(self.norm1(x), h) #
        x = x + x_
        return x, h

class SimpleViT(nn.Module):
    def __init__(self, in_dim, d_model, out_dim=None, n_heads=4, nlyrs=1):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        # self.embed = nn.Sequential( # in, out, kernel, stride, pad
        #     nn.Conv2d(in_dim, d_model, 7, 2, 7//2, bias=False), nn.BatchNorm2d(d_model), nn.ReLU(),
        #     nn.Conv2d(d_model, d_model, 3, 2, 3//2, bias=False)
        #     )
        self.blk = Seq(*[Block(d_model, n_heads) for _ in range(nlyrs)])
        self.attn_pool = nn.Linear(d_model, 1)
        self.out = nn.Linear(d_model, out_dim or d_model, bias=False)

    def forward(self, x, mask=None): # [b,c,h,w]
        x = x.flatten(2).transpose(1,2) # [b,h*w,c]
        x = self.embed(x)
        # print('vit fwd', x.shape)
        x,_ = self.blk(x)
        attn = self.attn_pool(x) # [b,h*w,1] # seq_pool
        x = (attn.softmax(dim=1).transpose(-2,-1) @ x).squeeze(1) # [b,1,h*w] @ [b,h*w,d] -> [b,d]
        return self.out(x)

# # b,t,d = 2,256,16
# b,t,d = 2,7,16
# x = torch.rand(b,t,d, device=device)
# model = Block(d_model=d, n_heads=4).to(device)
# # model = Seq(*[Block(d_model=d, n_heads=4) for _ in range(2)])
# out, h = model(x)
# print(out.shape)

dim = 64#64
in_dim=3
out_dim = 10
model = SimpleViT(in_dim, 64, out_dim, nlyrs=1, n_heads=8).to(device) # 64129
print(sum(p.numel() for p in model.parameters() if p.requires_grad)) # 59850
optim = torch.optim.AdamW(model.parameters(), lr=1e-3)

# print(images.shape) # [batch, 3, 32, 32]
x = torch.rand(24, 3, 32,32, device=device)
# x = torch.rand(64, 3, 28,28, device=device)
logits = model(x)
print(logits.shape)

# hydra lin 47257p Accuracy: 54.1%, Avg loss: 1.276877 time: 1.9073486328125e-06 118.25601513385773
# hydra conv 93401p Accuracy: 69.5%, Avg loss: 0.896258 time: 1.6689300537109375e-06 17.178589010238646
# trans ggrope lin 50945p Accuracy: 50.6%, Avg loss: 1.364122 time: 1.6689300537109375e-06 26.679788732528685
# trans ggrope conv 97089p Accuracy: 67.3%, Avg loss: 0.948646 time: 1.6689300537109375e-06 17.618668341636656

# Hydra one BCdt lin 37529p
# Hydra one BCdt expand3 lin 50905p Accuracy: 54.8%, Avg loss: 1.263319 time: 1.6689300537109375e-06 123.11688303947449

# hydra conv 83673


50889
torch.Size([24, 10])


In [6]:
# @title wandb
!pip install -q wandb
import wandb # https://docs.wandb.ai/quickstart
wandb.login(key='487a2109e55dce4e13fc70681781de9f50f27be7')
try: run.finish()
except NameError: pass
run = wandb.init(project="vit", config={"model": "hydra",})

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bobdole to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# @title train test function
import torch.nn.functional as F
# from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter()
# global_step=0

def train(dataloader, model, optim):
    size = len(dataloader.dataset)
    model.train()
    for i, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = F.cross_entropy(pred, y)
        optim.zero_grad()
        loss.backward()
        optim.step()
        if i % (size//(10* len(x))) == 0:
            loss, current = loss.item(), i * len(x)
            # loss_list.append(loss)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        try: wandb.log({"loss": loss.item()})
        except: pass
        # global global_step
        # writer.add_scalars('train', {"loss": loss}, global_step=global_step, walltime=None) # https://docs.pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_scalar
        # global_step+=1

def test(dataloader, model):
    model.eval()
    test_loss, correct = 0, 0
    for X, y in dataloader:
        x, y = X.to(device), y.to(device)
        with torch.no_grad(): pred = model(x)
        test_loss += F.cross_entropy(pred, y).item()
        correct += (pred.argmax(1) == y).sum().item()
    test_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    try: wandb.log({"test_loss": test_loss.item()})
    except: pass
    # global global_step
    # writer.add_scalars('test', {"acc": 100*correct, "loss": test_loss}, global_step=global_step, walltime=None) # https://docs.pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_scalar
    return correct

import time
start = time.time()
for t in range(10):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, optim)
    test(test_loader, model)
    end = time.time()
    print('time:',time.time() - end, (time.time()-start)/(t+1))

end = time.time()
print("time: ",end - start)
# writer.close()


Epoch 1
-------------------------------
loss: 2.341299  [    0/50000]
loss: 2.006038  [ 4992/50000]
loss: 1.919251  [ 9984/50000]
loss: 1.878052  [14976/50000]
loss: 1.850179  [19968/50000]
loss: 1.798870  [24960/50000]
loss: 1.848886  [29952/50000]
loss: 1.867756  [34944/50000]
loss: 1.915994  [39936/50000]
loss: 1.705564  [44928/50000]
Test Error: 
 Accuracy: 35.0%, Avg loss: 1.792743 

time: 2.384185791015625e-06 127.08035373687744
Epoch 2
-------------------------------
loss: 1.809276  [    0/50000]
loss: 1.846378  [ 4992/50000]
loss: 1.787947  [ 9984/50000]
loss: 1.845508  [14976/50000]
loss: 1.791700  [19968/50000]
loss: 1.621732  [24960/50000]
loss: 1.719482  [29952/50000]
loss: 1.638521  [34944/50000]
loss: 1.808813  [39936/50000]
loss: 1.731848  [44928/50000]
Test Error: 
 Accuracy: 38.6%, Avg loss: 1.699719 

time: 1.6689300537109375e-06 126.77289140224457
Epoch 3
-------------------------------
loss: 1.654131  [    0/50000]
loss: 1.695986  [ 4992/50000]
loss: 1.546870  [ 998

In [ ]:
# @title save/load
from google.colab import drive
drive.mount('/content/drive')
folder='/content/drive/MyDrive/jepa/'

# modelsd, optimsd = torch.load(folder+'hydra.pkl', map_location=device).values()
# model.load_state_dict(modelsd, strict=False)
# optim.load_state_dict(optimsd)

In [ ]:
checkpoint = {'model': model.state_dict(), 'optimizer': optim.state_dict()}
torch.save(checkpoint, folder+'hydra.pkl')
# torch.save(checkpoint, 'hydra.pkl')

### store

In [ ]:
# @title hydra me
# https://github.com/goombalab/hydra/blob/main/hydra/modules/hydra.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class Hydra(nn.Module):
    # def __init__(self, d_model, expand=2, n_heads=8, n_groups=1, d_state=64, d_conv=4):
    def __init__(self, d_model, expand=2, n_heads=8, n_groups=8, d_state=8, d_conv=7):
        super().__init__()
        self.d_model, self.n_groups, self.d_state, self.d_conv = d_model, n_groups, d_state, d_conv
        self.d_inner = expand * self.d_model
        self.n_heads, self.d_head = n_heads, self.d_inner//n_heads

        self.in_proj = nn.Linear(self.d_model, 2* self.d_inner + 2* (2*self.n_groups*self.d_state) + 2* self.n_heads, bias=False) # z,x,B,C,dt
        conv_dim = self.d_inner + 2* (2*self.n_groups*self.d_state) # for x,B,C
        # print(d_conv, conv_dim)
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=d_conv, groups=conv_dim, padding=d_conv//2, bias=True)
        # self.conv1d.weight._no_weight_decay = True
        self.act = nn.SiLU()

        # self.h0 = nn.Parameter(torch.zeros(self.n_heads, self.d_head, self.d_state))
        # self.h0._no_weight_decay = True
        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.n_heads) * (math.log(dt_max)-math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_weight_decay = True
        # A = torch.empty(self.n_heads, dtype=torch.float32).uniform_(1,16)
        A = torch.ones(self.n_heads, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A))
        self.A_log._no_weight_decay = True
        # self.D = nn.Parameter(torch.ones(self.n_heads))
        # self.D._no_weight_decay = True
        # self.fc_D = nn.Linear(self.d_inner, self.n_heads, bias=False)

        self.D = nn.Linear(self.d_inner, self.n_heads)
        self.D.bias._no_weight_decay = True

        self.norm = nn.RMSNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(self, u, h=None): # [b,t,d]
        b = u.shape[0]
        # print('Hydra u', u.shape)
        A = -torch.exp(self.A_log) # [n_heads]
        z, xBC, dt = self.in_proj(u).split([self.d_inner, self.d_inner + 2* (2*self.n_groups*self.d_state), 2* self.n_heads], dim=-1)

        dt = torch.cat((dt[...,:self.n_heads], torch.flip(dt[...,self.n_heads:], (1,))), dim=0) # [b,t,2h]->[2b,t,h]
        dt = F.softplus(dt+self.dt_bias) # [2b,t,h]+[h]

        xBC = self.act(self.conv1d(xBC.transpose(-2,-1)).transpose(-2,-1))  # [b,t, d_inner + 2* n_groups*d_state]
        # x, B, C = xBC.split([self.n_heads*self.d_head, self.n_groups*self.d_state, self.n_groups*self.d_state], dim=-1) # x:[b,t,h,d], B/C:[b,t,g,s] # X, B, C correspond to V, K, Q respectively in the SSM/attention duality
        x, BC = xBC.split([self.d_inner, 2* (2*self.n_groups*self.d_state)], dim=-1)
        x_og = x
        x = torch.cat((x, torch.flip(x,(1,))), dim=0).unflatten(-1, (self.n_heads, self.d_head)) # [2b,t,inr]->[2b,t,h,d]
        B, C = torch.cat((BC[...,:2*self.n_groups*self.d_state], torch.flip(BC[...,2*self.n_groups*self.d_state:], (1,))), dim=0).chunk(2, dim=-1) # [b,t,2*2gs]->[2b,t,2gs]->[2b,t,gs]
        B, C = B.unflatten(-1, (self.n_groups, self.d_state)), C.unflatten(-1, (self.n_groups, self.d_state)) # x:[2b,t,h,d], B/C:[2b,t,g,s]
        h_g = self.n_heads//self.n_groups
        if h_g>1: B, C = B.repeat_interleave(h_g, dim=-2), C.repeat_interleave(h_g, dim=-2) # [b,g,s]->[b,h,s]

        # h0 = self.h0.expand(u.size(0),-1,-1,-1) # [b,n,d,s]
        # print('x dt a b', x.shape, dt.shape, A.shape, B.shape)
        y, h = ssd(x*dt.unsqueeze(-1), A*dt, B, C, 64, h) # 256

        # y = y + x * self.D.unsqueeze(-1)
        # # y = self.norm(y.flatten(2) * self.act(z)) # [b,t,d_inner] # norm(x)*silu(z) if norm_before_gate, else norm(x*silu(z)) # https://github.com/state-spaces/mamba/blob/main/mamba_ssm/ops/triton/layernorm_gated.py#L18

        y = y.flatten(2) # [2b,t,inr]
        y = torch.roll(y, shifts=1, dims=1) # 123...l -> l12...l-1
        y[:,0] = 0. # 012...l-1

        # y = y[:b] + torch.flip(y[b:], (1,)) + x_og * F.linear(x_og, self.fc_D.weight, bias=self.D).repeat(1,1,self.d_head)
        y = y[:b] + torch.flip(y[b:], (1,)) + x_og * self.D(x_og).repeat(1,1,self.d_head) # [b,t,h*inr]

        y = self.norm(y.flatten(2)) * self.act(z)
        out = self.out_proj(y)
        return out, h # [b,t,in]


b,t,d_model=5,256,32
# b,t,d_model=5,7,32
d_model = 64
u = torch.randn(b,t,d_model, device=device)
model = Hydra(d_model).to(device)
out, h = model(u)
# h0 = torch.randn(b, model.n_heads, model.d_head, model.d_state)
# print(out.shape)
# print(out.shape, h.shape)
# print(out[0,-3:,:5], h[0,:2,:5,:5])
# out, h = model(u, h)

# u = torch.randn(b,7,d_model, device=device)
# # out, h = model(u[:,-1:], h)
# out, h = model.step(u, h)
# out, h = model.step(u)
print(out.shape)
# print(out[0,-3:,:5], h[0,:2,:5,:5])



In [ ]:
# @title goombalab hydra.py
# https://github.com/goombalab/hydra/blob/main/hydra/modules/hydra.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange, repeat
try:
    from mamba_ssm.ops.triton.layernorm_gated import RMSNorm as RMSNormGated
except ImportError:
    RMSNormGated = None
from mamba_ssm.ops.triton.ssd_combined import mamba_chunk_scan_combined
from hydra.modules.ops import hydra_split_conv1d_scan_combined


class Hydra(nn.Module):
    def __init__(
        self,
        d_model,
        d_state=64,
        d_conv=7,
        conv_init=None,
        expand=2,
        headdim=64,
        ngroups=1,
        dt_min=0.001,
        dt_max=0.1,
        dt_init_floor=1e-4,
        dt_limit=(0.0, float("inf")),
        learnable_init_states=False,
        activation="swish",
        bias=False,
        conv_bias=True,
        # Fused kernel and sharding options
        chunk_size=256,
        layer_idx=None,  # Absorb kwarg for general module
        device=None,
        dtype=None,
    ):
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.conv_init = conv_init
        self.expand = expand
        self.d_inner = self.expand * self.d_model
        self.headdim = headdim
        self.ngroups = ngroups
        assert self.d_inner % self.headdim == 0
        self.nheads = self.d_inner // self.headdim
        self.dt_limit = dt_limit
        self.learnable_init_states = learnable_init_states
        self.activation = activation
        self.chunk_size = chunk_size
        self.layer_idx = layer_idx

        # Order: [z, x, B, C, dt]
        d_in_proj = 2 * self.d_inner + 2 * (2 * self.ngroups * self.d_state) + 2 * self.nheads
        self.in_proj = nn.Linear(self.d_model, d_in_proj, bias=bias, **factory_kwargs)

        conv_dim = self.d_inner + 2 * (2 * self.ngroups * self.d_state)
        self.conv1d = nn.Conv1d(
            in_channels=conv_dim,
            out_channels=conv_dim,
            bias=conv_bias,
            kernel_size=d_conv,
            groups=conv_dim,
            padding=d_conv // 2,
            **factory_kwargs,
        )
        if self.conv_init is not None:
            nn.init.uniform_(self.conv1d.weight, -self.conv_init, self.conv_init)
        # self.conv1d.weight._no_weight_decay = True

        if self.learnable_init_states:
            self.init_states = nn.Parameter(torch.zeros(self.nheads, self.headdim, self.d_state, **factory_kwargs))
            self.init_states._no_weight_decay = True

        self.act = nn.SiLU()

        # Initialize log dt bias
        dt = torch.exp(torch.rand(self.nheads, **factory_kwargs) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min))
        dt = torch.clamp(dt, min=dt_init_floor)
        # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        self.dt_bias = nn.Parameter(inv_dt)
        # Just to be explicit. Without this we already don't put wd on dt_bias because of the check
        # name.endswith("bias") in param_grouping.py
        self.dt_bias._no_weight_decay = True

        # A parameter
        A = torch.ones(self.nheads, dtype=torch.float32, device=device)
        A_log = torch.log(A).to(dtype=dtype)
        self.A_log = nn.Parameter(A_log)
        # self.register_buffer("A_log", torch.zeros(self.nheads, dtype=torch.float32, device=device), persistent=True)
        self.A_log._no_weight_decay = True

        # D "skip" parameter
        self.D = nn.Parameter(torch.ones(self.nheads, device=device))
        self.D._no_weight_decay = True
        self.fc_D = nn.Linear(self.d_inner, self.nheads, bias=False, **factory_kwargs)

        # Extra normalization layer right before output projection
        assert RMSNormGated is not None
        self.norm = RMSNormGated(self.d_inner, eps=1e-5, norm_before_gate=True, **factory_kwargs)

        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=bias, **factory_kwargs)

    def forward(self, u, seq_idx=None): # [b,l,d]
        batch, seqlen, dim = u.shape
        zxbcdt = self.in_proj(u)  # (B, L, d_in_proj)
        A = -torch.exp(self.A_log.float())  # (nheads) or (d_inner, d_state)
        initial_states = repeat(self.init_states, "... -> b ...", b=2*batch) if self.learnable_init_states else None
        dt_limit_kwargs = {} if self.dt_limit == (0.0, float("inf")) else dict(dt_limit=self.dt_limit)
        z, xBC, dt = torch.split(zxbcdt, [self.d_inner, self.d_inner + 2 * (2 * self.ngroups * self.d_state), 2 * self.nheads], dim=-1)

        dt = torch.cat((dt[:, :, :self.nheads], torch.flip(dt[:, :, self.nheads:], (1,))), dim=0)
        dt = F.softplus(dt + self.dt_bias)  # (2 * B, L, nheads)
        assert self.activation in ["silu", "swish"]

        # 1D Convolution
        xBC = self.act(
            self.conv1d(xBC.transpose(1, 2)).transpose(1, 2)
        )  # (B, L, self.d_inner + 2 * (2 * ngroups * d_state))

        # Split into 3 main branches: X, B, C
        # These correspond to V, K, Q respectively in the SSM/attention duality
        x, BC = torch.split(xBC, [self.d_inner, 2 * (2 * self.ngroups * self.d_state)], dim=-1)
        x_og = x
        x = torch.cat((x, torch.flip(x, (1,))), dim=0)
        BC = torch.cat(
            (BC[:, :, :2 * self.ngroups * self.d_state],
             torch.flip(BC[:, :, 2 * self.ngroups * self.d_state:], (1,))),
            dim=0
        )
        B, C = torch.split(BC, [self.ngroups * self.d_state, self.ngroups * self.d_state], dim=-1)

        y = mamba_chunk_scan_combined(
            rearrange(x, "b l (h p) -> b l h p", p=self.headdim),
            dt,
            A,
            rearrange(B, "b l (g n) -> b l g n", g=self.ngroups),
            rearrange(C, "b l (g n) -> b l g n", g=self.ngroups),
            chunk_size=self.chunk_size,
            D=None,
            z=None,
            seq_idx=seq_idx,
            initial_states=initial_states,
            **dt_limit_kwargs,
        )
        y = rearrange(y, "b l h p -> b l (h p)")
        y = torch.roll(y, shifts=1, dims=1)
        y[:, 0, :] = 0.0
        y_fw, y_bw = y[:batch], torch.flip(y[batch:], (1,))
        y = y_fw + y_bw + x_og * repeat(
            F.linear(x_og, self.fc_D.weight, bias=self.D), "b l h -> b l (h p)", p=self.headdim
        )

        # Multiply "gate" branch and apply extra normalization layer
        y = self.norm(y, z)
        out = self.out_proj(y)
        return out


In [ ]:
# @title goombalab matrix_mixers/quasiseparable.py
# https://github.com/goombalab/hydra/blob/main/hydra/modules/matrix_mixers/quasiseparable.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class Quasiseparable(nn.Module):
    # NOTE Data-dependent quasiseparable is equivalent to Hydra that shares BC for forward and reverse sequence processing.
    def __init__(self,
        is_data_dependent,
        d_model, qk_dim, expand=2, headdim=128,
        max_seq_len=None, # max_seq_len is necessary for data-independent version.
    ):
        super().__init__()
        self.is_data_dependent = is_data_dependent
        self.d_model, self.qk_dim, self.expand = d_model, qk_dim, expand
        self.max_seq_len = max_seq_len
        self.d_inner = self.expand*self.d_model
        # assert self.d_inner % self.headdim == 0
        self.nheads, self.headdim = self.d_inner//headdim, headdim
        self.d_state = self.nheads*qk_dim

        if not self.is_data_dependent:
            self.BC = nn.Parameter(torch.empty(self.max_seq_len, 2* self.d_state))
            nn.init.xavier_normal_(self.BC)

        # Initialize log dt bias
        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.nheads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min))
        dt = torch.clamp(dt, min=1e-4)
        # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        self.dt_bias = nn.Parameter(inv_dt)
        # Just to be explicit. Without this we already don't put wd on dt_bias because of the check
        # name.endswith("bias") in param_grouping.py
        self.dt_bias._no_weight_decay = True

        # A parameter
        A = torch.ones(self.nheads, dtype=torch.float32, device=device)
        self.A_log = nn.Parameter(torch.log(A).to(dtype=dtype))
        # self.register_buffer("A_log", torch.zeros(self.nheads, dtype=torch.float32, device=device), persistent=True)
        self.A_log._no_weight_decay = True

        # D "skip" parameter
        self.D = nn.Parameter(torch.ones(self.nheads, device=device))
        self.D._no_weight_decay = True
        self.fc_D = nn.Linear(self.d_inner, self.nheads, bias=False)

    def forward(self, x, BC, dt):
        batch, seqlen, dim = x.shape
        A = -torch.exp(self.A_log.float())  # (nheads) or (d_inner, d_state)
        dt = torch.cat((dt[:, :, :self.nheads], torch.flip(dt[:, :, self.nheads:], (1,))), dim=0)
        dt = F.softplus(dt) # (2 * B, L, nheads)

        x_og = x
        x = torch.cat((x, torch.flip(x, (1,))), dim=0)
        if BC.dtype != x.dtype:
            BC = BC.to(x.dtype)
        BC = torch.cat((BC, torch.flip(BC, (1,))), dim=0)
        B, C = torch.split(BC, [self.d_state, self.d_state], dim=-1)

        y = mamba_chunk_scan_combined(
            rearrange(x, "b l (h p) -> b l h p", p=self.headdim),
            dt,
            A,
            rearrange(B, "b l (g n) -> b l g n", g=1),  # Fixing ngroups to 1 for simplicity.
            rearrange(C, "b l (g n) -> b l g n", g=1),  # Fixing ngroups to 1 for simplicity.
            chunk_size=256,
            D=None,
            z=None,
            seq_idx=None,
            initial_states=None,
        )
        y = rearrange(y, "b l h p -> b l (h p)")
        y = torch.roll(y, shifts=1, dims=1)
        y[:, 0, :] = 0.0
        y_fw, y_bw = y[:batch], torch.flip(y[batch:], (1,))
        y = y_fw + y_bw + x_og * repeat(
            F.linear(x_og, self.fc_D.weight, bias=self.D), "b l h -> b l (h p)", p=self.headdim
        )
        return y


In [ ]:
# @title goombalab matrix_mixer.py
# https://github.com/goombalab/hydra/blob/main/hydra/modules/matrix_mixer.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MatrixMixer(nn.Module):
    def __init__(self,
        is_data_dependent, # True?
        d_model, qk_dim, expand=2, headdim=128):
        super().__init__()
        self.is_data_dependent = is_data_dependent
        self.d_model, self.qk_dim, self.expand = d_model, qk_dim, expand
        self.d_inner = self.expand * self.d_model
        # assert self.d_inner % self.headdim == 0
        self.nheads, self.headdim = self.d_inner//headdim, headdim
        self.d_state = self.nheads*qk_dim

        # Order: [z, x, B, C, dt]
        d_in_proj = 2 * self.d_inner + self.is_data_dependent * (2 * self.d_state + 2 * self.nheads)
        conv_dim = self.d_inner + self.is_data_dependent * (2 * self.d_state)
        self.matrix_mixer = Quasiseparable(self.is_data_dependent, self.d_model, self.qk_dim, max_seq_len=None, expand=self.expand, headdim=self.headdim, chunk_size=self.chunk_size)

        self.in_proj = nn.Linear(self.d_model, d_in_proj, bias=False)
        d_conv = 7
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=d_conv, groups=conv_dim, padding=d_conv//2, bias=True)
        # nn.init.uniform_(self.conv1d.weight, -self.conv_init, self.conv_init)
        # self.conv1d.weight._no_weight_decay = True

        self.act = nn.SiLU()
        # Extra normalization layer right before output projection
        assert RMSNormGated is not None
        self.norm = RMSNormGated(self.d_inner, eps=1e-5, norm_before_gate=True)
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(self, u): # [b,t,d]
        batch, seqlen, dim = u.shape
        assert self.activation in ["silu", "swish"]
        u_proj = self.in_proj(u)  # (B, L, d_in_proj)

        # elif self.matrix_mixer_type == "quasiseparable":
        if self.is_data_dependent:
            z, vqk, dt = torch.split(u_proj, [self.d_inner, self.d_inner + 2* self.d_state, 2* self.nheads], dim=-1)
            dt = dt + repeat(self.matrix_mixer.dt_bias, 'h -> (2 h)')
            vqk = self.act(self.conv1d(vqk.transpose(1, 2)).transpose(1, 2))
            v, qk = torch.split(vqk, [self.d_inner, 2* self.d_state], dim=-1)
            y = self.matrix_mixer(v, qk, dt)
        else:
            z, v = torch.split(u_proj, [self.d_inner, self.d_inner], dim=-1)
            dt = repeat(self.matrix_mixer.dt_bias, 'h -> b l (2 h)', b=batch, l=seqlen)
            v = self.act(self.conv1d(v.transpose(1, 2)).transpose(1, 2))
            qk = repeat(self.matrix_mixer.BC, 'l d -> b l d', b=batch)
            y = self.matrix_mixer(v, qk, dt)

        # Multiply "gate" branch and apply extra normalization layer
        y = self.norm(y, z)
        out = self.out_proj(y)
        return out
